# Load improved model

Choose a saved improved run, load its checkpoint, verify save/load behavior, plot training curves, and inspect decoded validation predictions.


In [ ]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import tempfile
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np

from src.data.freihand import FreiHand, SPLIT_SEED, SPLIT_VALIDATION_FRACTION
from src.evaluation.metrics import evaluate_model, sample_mpke
from src.evaluation.overlays import prediction_grid, save_figure
from src.evaluation.training_curves import load_history, plot_training_curves

from src.models.heatmaps import wrap_with_keypoint_decoder


## Choose run


In [ ]:
IMPROVED_RUN = 'improved-model-1'
AVAILABLE_IMPROVED = ('improved-model-1',)

if IMPROVED_RUN not in AVAILABLE_IMPROVED:
    raise ValueError(f'Unknown improved run: {IMPROVED_RUN}')

run_dir = project_root / 'models' / IMPROVED_RUN
checkpoint_path = run_dir / 'best.keras'
config_path = project_root / 'logs' / IMPROVED_RUN / 'config.json'
evaluation_path = project_root / 'artifacts' / IMPROVED_RUN / 'evaluation.json'

if not config_path.exists():
    raise FileNotFoundError(f'Missing run config: {config_path}')
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')

config = json.loads(config_path.read_text())
input_size = int(config.get('input_shape', [224])[0])
print(f'Run:        {IMPROVED_RUN}')
print(f'Model ID:   {config.get("model_id", config.get("model"))}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Input size: {input_size}')
print(json.dumps(config, indent=2))


## Training curves


In [ ]:
history = load_history(IMPROVED_RUN)
fig = plot_training_curves(history, suptitle=f'Improved training: {IMPROVED_RUN}')
saved_path = save_figure(fig, f'{IMPROVED_RUN}_training_curves')
print(f'Saved figure: {saved_path.relative_to(project_root)}')
plt.show()


## Load and verify checkpoint


In [ ]:
model = keras.models.load_model(checkpoint_path)
model.summary()

dummy = np.random.default_rng(0).random((4, input_size, input_size, 3), dtype=np.float32)
expected = model.predict(dummy, verbose=0)

with tempfile.TemporaryDirectory() as tmp:
    roundtrip_path = Path(tmp) / 'roundtrip.keras'
    model.save(roundtrip_path)
    reloaded = keras.models.load_model(roundtrip_path)
    actual = reloaded.predict(dummy, verbose=0)

assert np.array_equal(expected, actual), 'Round-trip predictions do not match'
print('Round-trip OK: reloaded model produces identical raw outputs.')

eval_model = wrap_with_keypoint_decoder(model, input_size=input_size) if len(model.output_shape) == 4 else model


## Validation sample overlays


In [ ]:
dataset = FreiHand()
dataset.validate()
_, val_idx = dataset.train_validation_split(
    validation_fraction=SPLIT_VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)

sample_ids = val_idx[:6]
images, ground_truth = dataset.load_batch(
    sample_ids,
    image_size=(input_size, input_size),
    flatten_keypoints=False,
)
predictions = eval_model.predict(images, verbose=0).reshape(-1, 21, 2)

fig = prediction_grid(
    images,
    ground_truth,
    predictions,
    ncols=3,
    suptitle=f'{IMPROVED_RUN}: validation predictions',
)
plt.show()


## Full validation metric


In [ ]:
val_ds = dataset.tf_dataset(
    indices=val_idx,
    batch_size=64,
    image_size=(input_size, input_size),
    flatten_keypoints=True,
)
metrics = evaluate_model(eval_model, val_ds, include_representative_indices=True)
print(metrics)


## Representative samples


In [ ]:
if evaluation_path.exists():
    evaluation = json.loads(evaluation_path.read_text())
    representative = evaluation['metrics']['representative_indices']
else:
    representative = metrics['representative_indices']

labels = ['best', 'median', 'p90', 'worst']
sample_ids = np.array([val_idx[representative[label]] for label in labels])
images, ground_truth = dataset.load_batch(
    sample_ids,
    image_size=(input_size, input_size),
    flatten_keypoints=False,
)
predictions = eval_model.predict(images, verbose=0).reshape(-1, 21, 2)
errors = sample_mpke(predictions, ground_truth)
titles = [f'{label} ({error:.1f} px)' for label, error in zip(labels, errors)]

fig = prediction_grid(
    images,
    ground_truth,
    predictions,
    titles=titles,
    ncols=2,
    suptitle=f'{IMPROVED_RUN}: representative predictions',
)
saved_path = save_figure(fig, f'{IMPROVED_RUN}_representative_predictions')
print(f'Saved figure: {saved_path.relative_to(project_root)}')
plt.show()
